# Predicting football match outcomes utilising machine learning

The aim of this project is to predict whether the home or away team will win using historic match data and machine leaning models.

- Import libraries

In [ ]:
import pandas as pd
import datetime as dt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
import seaborn as sns

- Load dataset

This dataset contains English premier league match data from 2000 untill the 2024/25 season.

In [ ]:
df = pd.read_csv("data/epl.csv")
df.head()

In [ ]:
df.info() # checking for how many columns are presnet 
df.shape
df.isnull().sum()

* change string matchdate to an actual date so we can process in chronological order

In [ ]:
df["MatchDate"] = pd.to_datetime(df["MatchDate"], format = "mixed")
df = df.sort_values("MatchDate")
df["MatchDate"]

In [ ]:
df["Year"] = df["MatchDate"].dt.year
df["Month"] = df["MatchDate"].dt.month # helps models pick up seaonal patterns

- Feature engineering.

  calculate what columns to use for feature engineering (preferably not stats after match as that could cause data leakage and make our model invalid as we want to predict post game results).

  we should have:

  average goals conceded (home and away),
  goal difference,
  shots per game,
  form in the last 5 matches (win rate)

In [ ]:
df["HomeWins"] = (df["FullTimeHomeGoals"] > df["FullTimeAwayGoals"]).astype(int) # .astype changes boolean to integer 
df["HomeWins"]

In [ ]:
df["AwayWins"] = (df["FullTimeAwayGoals"] > df["FullTimeHomeGoals"]).astype(int)
df["AwayWins"]

In [ ]:
df["HomeFormInLast5"] = (df.groupby("HomeTeam")["HomeWins"].transform(lambda x : x.shift().rolling(5).mean()))
df["AwayFormInLast5"] = (df.groupby("AwayTeam")["AwayWins"].transform(lambda x : x.shift().rolling(5).mean()))

In [ ]:
df["AwayConcededInLast5"] = (df.groupby("AwayTeam")["FullTimeHomeGoals"].transform(lambda x : x.shift().rolling(5).mean()))
df["HomeConcededInLast5"] = (df.groupby("HomeTeam")["FullTimeAwayGoals"].transform(lambda x : x.shift().rolling(5).mean()))


In [ ]:
df["AvgHGoals"] = (df.groupby("HomeTeam")["FullTimeHomeGoals"].transform(lambda x: x.shift().rolling(5).mean()))
df["AvgAGoals"] = (df.groupby("AwayTeam")["FullTimeAwayGoals"].transform(lambda x: x.shift().rolling(5).mean()))
df["AvgHGoals"]

In [ ]:
df["HGoalDiff"] = df["FullTimeHomeGoals"] - df["FullTimeAwayGoals"]
df["AGoalDiff"] = df["FullTimeAwayGoals"] - df["FullTimeHomeGoals"]

df["HGoalDiffLast5"] = df.groupby("HomeTeam")["HGoalDiff"].transform(lambda x: x.shift().rolling(5).mean())
df["AGoalDiffLast5"] = df.groupby("AwayTeam")["AGoalDiff"].transform(lambda x: x.shift().rolling(5).mean())


- Making one big difference rather than having home and away 

In [ ]:
# if positive home negative in favour of away
df["GoalDiffLast5"] = df["HGoalDiffLast5"] - df["AGoalDiffLast5"]  
df["AvgGoalsLast5"] = df["AvgHGoals"] - df["AvgAGoals"]
df["GoalsConcededLast5"] = df["HomeConcededInLast5"] - df["AwayConcededInLast5"]
df["FormInLast5"] = df["HomeFormInLast5"] - df["AwayFormInLast5"]

df = df.dropna()
df.isna().sum()



- Test/Train split 

In [ ]:
x = df[["GoalDiffLast5", "AvgGoalsLast5", "GoalsConcededLast5", "FormInLast5"]]
y = df["HomeWins"]

X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42) 

In [ ]:
# scale features for Knn as magnitude is sensitive 
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


 - Knn (Distance model)

In [ ]:
# Train model
from sklearn.neighbors import KNeighborsClassifier
for k in range(1,30):
 knn = KNeighborsClassifier(n_neighbors = k)
 knn.fit(X_train, y_train)
 print("K-value :", k, ", Accuracy :",knn.score(X_test, y_test)) 
 

Used parameter hypertuning and discovered that larger values of k increase accuracy where the best performance monitored was k = 24 where accuracy was 60.2 percent however, I have also noticed the peak drops off after k = 24 

In [ ]:
knn = KNeighborsClassifier(n_neighbors = 24)
knn.fit(X_train,y_train)
knn.score(X_test, y_test)

- Random forest (tree model)
+ feature importance to indicate which feature is heavily referenced


In [ ]:
rf = RandomForestClassifier(n_estimators=200, max_depth=10 ,random_state=42) # tuned by adding increaesed number of estimators(trees) and depth
rf.fit(X_train, y_train)
rf.score(X_test, y_test)

In [ ]:
feature_importance = rf.feature_importances_
features = x.columns

pd.Series(feature_importance, index= features).sort_values().plot(kind="barh")
plt.title("Feature importance")
plt.show()

 - base model logistic regression

In [ ]:
Lreg = LogisticRegression(max_iter=1000)
Lreg.fit(X_train,y_train)
Lreg.score(X_test, y_test)


- create confusion matrices and used heatmaps for each classification model in order to see how accurate each prediction was for home and away games 

In [ ]:
Log_predictions = Lreg.predict(X_test)
Log_cm = confusion_matrix(y_test, Log_predictions)

Log_cm_df = pd.DataFrame(Log_cm,
                     index=["HomeWin", "AwayWin"],
                     columns=["HomeWin", "AwayWin"])

rf_predictions = rf.predict(X_test)
rf_cm = confusion_matrix(y_test, rf_predictions)

rf_cm_df = pd.DataFrame(rf_cm,
                     index=["HomeWin", "AwayWin"],
                     columns=["HomeWin", "AwayWin"])


knn_predictions = knn.predict(X_test)
knn_cm = confusion_matrix(y_test, knn_predictions)

knn_cm_df = pd.DataFrame(knn_cm,
                     index=["HomeWin", "AwayWin"],
                     columns=["HomeWin", "AwayWin"])





In [ ]:
rf_cm_df

In [ ]:
knn_cm_df

In [ ]:
Log_cm_df   

In [ ]:
sns.heatmap(Log_cm, annot=True, fmt="d", cmap = "Blues",
             xticklabels=["Home Win","Away Win"],yticklabels=["Home Win","Away Win"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Log_Reg Confusion Matrix")
plt.show()

In [ ]:
sns.heatmap(rf_cm, annot=True, fmt="d", cmap = "Greens",
             xticklabels=["Home Win","Away Win"],yticklabels=["Home Win","Away Win"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Rf Confusion Matrix")
plt.show()

In [ ]:
sns.heatmap(knn_cm, annot=True, fmt="d", cmap = "Reds",
             xticklabels=["Home Win","Away Win"],yticklabels=["Home Win","Away Win"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("knn Confusion Matrix")
plt.show()